# 01 – Scielo Crawler: Coleta de Artigos do JMOe

## Resumo
Realiza o crawling automatizado dos artigos publicados nas edições do *Journal of
Microwaves, Optoelectronics and Electromagnetic Applications* (JMOe) no portal SciELO.

**Fluxo:**
1. Acessa a URL de uma edição.
2. Coleta os links individuais de cada artigo.
3. Extrai metadados (título, autores, palavras-chave, abstract, referências).
4. Persiste no MongoDB local (evita duplicatas automaticamente).

**Entrada:** URL da edição (`edition_url`) + MongoDB rodando em `localhost:27017`.  
**Saída:** Coleção `articles` no banco `scielo_db`.

> Ajuste `ano`, `volume` e `numero` no método `parse_articles` antes de rodar.


## Instalação de Dependências

In [ ]:
# requests-html: HTTP com suporte a XPath/JS | loguru: logging elegante | pymongo: driver MongoDB
!pip install requests-html loguru pymongo -q


## Imports e Seletores XPath

In [ ]:
import time
from typing import List, Optional

from loguru import logger
from pymongo import MongoClient
from pymongo.errors import DuplicateKeyError
from requests_html import HTMLSession

# Mapeamento XPath de cada campo do artigo no layout SciELO
# Ajuste os seletores se o layout do site mudar
XPATH_DATA = {
    "article_title":      "//*[@id='standalonearticle']/section/div/div/h1",
    "article_authors":    "/html/head/meta[@name='citation_author']/@content",
    "article_keywords":   "//*[@id='articleText']/div[1]/p[2]",
    "article_abstract":   "//*[@id='articleText']/div[1]/p[1]",
    "article_references": "//*[@id='articleText']/div/div/div/ul/li",
    "article_links":      "//*[@id='issueIndex']/div/div[2]/table/tbody/tr/td/ul/li[2]/a/@href",
}


## Classe ArticleCrawler

In [ ]:
class ArticleCrawler:
    """
    Crawler para uma edição do JMOe no portal SciELO.

    Parâmetros
    ----------
    mongo_uri       : URI de conexão MongoDB (padrão: localhost).
    db_name         : Banco de dados de destino.
    collection_name : Coleção onde os artigos serão salvos.
    """

    def __init__(self, mongo_uri='mongodb://localhost:27017',
                 db_name='scielo_db', collection_name='articles'):
        self.client = MongoClient(mongo_uri)
        self.db = self.client[db_name]
        self.collection = self.db[collection_name]
        # Índice único no campo 'link' previne duplicatas
        self.collection.create_index('link', unique=True)
        self.session = HTMLSession()

    # ------------------------------------------------------------------
    # Helpers privados de extração
    # ------------------------------------------------------------------

    def _safe_find_element(self, r, xpath: str) -> Optional[str]:
        """Retorna o texto do primeiro elemento encontrado pelo XPath, ou None."""
        try:
            elements = r.html.xpath(xpath)
            if elements:
                return elements[0] if isinstance(elements[0], str) else elements[0].text
            return None
        except Exception as e:
            logger.warning(f'Error finding element with xpath {xpath}: {e}')
            return None

    def _safe_find_elements(self, r, xpath: str) -> List[str]:
        """Retorna os textos de todos os elementos encontrados pelo XPath."""
        try:
            elements = r.html.xpath(xpath)
            return [elem.text if hasattr(elem,'text') else str(elem)
                    for elem in elements if elem]
        except Exception as e:
            logger.warning(f'Error finding elements with xpath {xpath}: {e}')
            return []

    def _extract_title(self, r) -> Optional[str]:
        """Extrai o título do artigo."""
        title = self._safe_find_element(r, XPATH_DATA['article_title'])
        return title.strip() if title else None

    def _extract_authors(self, r) -> str:
        """Extrai autores via metatag citation_author."""
        authors = r.html.xpath('//meta[@name="citation_author"]/@content')
        autores = [a.strip() for a in authors if a.strip()]
        return ', '.join(autores) if autores else ''

    def _extract_keywords(self, r) -> str:
        """Extrai palavras-chave e retorna como string separada por vírgulas."""
        kws = self._safe_find_elements(r, XPATH_DATA['article_keywords'])
        return ', '.join(k.strip() for k in kws if k.strip())

    def _extract_abstract(self, r) -> str:
        """Extrai o abstract do artigo."""
        abstract = self._safe_find_element(r, XPATH_DATA['article_abstract'])
        return abstract.strip() if abstract else ''

    def _extract_references(self, r) -> str:
        """Extrai referências bibliográficas, uma por linha."""
        refs = self._safe_find_elements(r, XPATH_DATA['article_references'])
        return '\n'.join(ref.strip() for ref in refs if ref.strip())

    # ------------------------------------------------------------------
    # Requisição HTTP com retry e espera exponencial
    # ------------------------------------------------------------------

    def _get_page(self, url: str, timeout: int = 300,
                  retries: int = 3, render: bool = False):
        """
        GET com retry automático. Define render=True para páginas com JavaScript.
        Espera exponencial entre tentativas: 1s, 2s, 4s...
        """
        for attempt in range(retries):
            try:
                r = self.session.get(url, timeout=timeout)
                r.raise_for_status()
                if render:
                    r.html.render(timeout=20)
                return r
            except Exception as e:
                logger.warning(f'Attempt {attempt+1} failed for {url}: {e}')
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    logger.error(f'Failed to fetch {url} after {retries} attempts')
                    raise

    # ------------------------------------------------------------------
    # Persistência
    # ------------------------------------------------------------------

    def _save_to_mongo(self, article: dict):
        """Insere artigo no MongoDB. Ignora duplicatas silenciosamente."""
        try:
            self.collection.insert_one(article)
            logger.success(f'Saved to MongoDB: {article["link"]}')
        except DuplicateKeyError:
            logger.info(f'Duplicate skipped: {article["link"]}')

    # ------------------------------------------------------------------
    # Métodos públicos
    # ------------------------------------------------------------------

    def collect_article_links(self, edition_url: str) -> List[str]:
        """Acessa página da edição e retorna lista de URLs absolutas dos artigos."""
        article_links = []
        try:
            r = self._get_page(edition_url, render=True)
            links = r.html.xpath(XPATH_DATA['article_links'])
            for href in links:
                if href:
                    article_links.append('https://www.scielo.br' + href)
        except Exception as e:
            logger.error(f'Error collecting article links from {edition_url}: {e}')
        return article_links

    def parse_articles(self, article_urls: List[str]):
        """
        Itera pelos URLs e extrai metadados.
        ⚠️  Altere ano, volume e numero abaixo conforme a edição.
        """
        parsed_counter = 0
        for i, url in enumerate(article_urls, start=1):
            try:
                logger.info(f'Parsing article {i}/{len(article_urls)}: {url}')
                r = self._get_page(url)
                time.sleep(1)  # Pausa educada ao servidor
                parsed_article = {
                    'ano':            '2025',  # ⚠️ Trocar manualmente
                    'volume':         '24',    # ⚠️ Trocar manualmente
                    'numero':         '1',     # ⚠️ Trocar manualmente
                    'titulo':         self._extract_title(r),
                    'autores':        self._extract_authors(r),
                    'palavras_chave': self._extract_keywords(r),
                    'abstract':       self._extract_abstract(r),
                    'referencias':    self._extract_references(r),
                    'link':           url,
                }
                self._save_to_mongo(parsed_article)
                parsed_counter += 1
                yield parsed_article
            except Exception as e:
                logger.error(f'Error parsing article {url}: {e}')
                continue
        logger.success(f'Parsed {parsed_counter} of {len(article_urls)} articles.')

    def crawl_edition(self, edition_url: str):
        """Pipeline completo: coleta links → extrai metadados → salva no MongoDB."""
        logger.info(f'Starting crawl for edition: {edition_url}')
        article_links = self.collect_article_links(edition_url)
        logger.info(f'Found {len(article_links)} articles in edition')
        return list(self.parse_articles(article_links))

    def close(self):
        """Fecha sessão HTTP e conexão MongoDB."""
        self.session.close()
        self.client.close()


## Execução

> Altere `edition_url` para a edição desejada.

In [ ]:
# Substitua pelo link da edição que deseja coletar
edition_url = 'https://www.scielo.br/j/jmoea/i/2025.v24n1/'

crawler = ArticleCrawler()
try:
    articles = crawler.crawl_edition(edition_url)
    logger.info(f'Crawl completed. Total articles processed: {len(articles)}')
finally:
    crawler.close()
